# Stage 2: Data Acquisition, Exploration, and Preprocessing

This task involves a deep dive into the dataset to identify linguistic patterns, class imbalances, and noise. If the noise can be cleaned easily, we ensure higher quality inputs for the binary classifiers and a more reliable training process downstream.

## Exploratory Data Analysis (EDA) in NLP

Exploratory Data Analysis (EDA) in NLP focuses on uncovering linguistic patterns, structural properties, biases, and noise within text data before model training.

Skipping EDA means training blindly. Performing it leads to better modeling decisions.

---

### 1. Basic Statistical Profiling

Understand dataset structure before examining content.

- **Token Count**  
  Average, minimum, and maximum sentence length; helps determine `max_length` for transformer models.

- **Vocabulary Size**  
  Number of unique words; influences embedding size and model capacity.

- **Class Distribution**  
  Check for imbalance; prevents misleading metrics (for example, high accuracy from majority class guessing).

Outcome: decide on class weighting, resampling, or augmentation, and set appropriate sequence length.

---

### 2. Lexical Analysis (Word-Level Patterns)

Examine the actual words used in the dataset.

- **N-gram Analysis**  
  Identify frequent bigrams/trigrams; detect domain-specific phrases.

- **Stopword Density**  
  Estimate filler content; inform cleaning strategy.

- **Word Frequency / Word Clouds**  
  Quick visual check to ensure frequent words align with the task intent.

Outcome: detect shortcuts the model might exploit and identify task-relevant keywords.

---

### 3. Semantic & Syntactic Exploration

Go beyond surface statistics into linguistic structure.

- **POS Tagging**  
  Analyze distribution of nouns, verbs, adjectives, etc., to identify stylistic patterns.

- **Named Entity Recognition (NER)**  
  Detect frequent entities such as people, locations, or organizations.

- **Embedding Visualization (t-SNE / UMAP)**  
  Project embeddings into 2D space to observe natural clustering patterns.

Outcome: understand deeper structure before training and identify potential bias or semantic grouping.

---

### 4. Identifying Noise & Artifacts

Remove data issues before modeling.

- **Duplicates**  
  Prevent data leakage.

- **Special Characters / HTML Artifacts**  
  Clean tokens that could break the tokenizer.

- **Outliers**  
  Extremely short or long texts may indicate data collection errors.

Outcome: improve training stability and prevent misleading evaluation results.

---

## Why EDA Is Critical

EDA helps determine:

1. Whether class balancing or augmentation is needed.
2. Appropriate sequence length (`max_length`).
3. Whether the task is artificially easy due to keyword shortcuts.

Performing EDA ensures informed model design rather than blind training.

## Data Loading & Preparation

Three sources:
- `dontpatronizeme_pcl.tsv` — full labeled corpus (par_id, keyword, country, text, binary label, orig_label 0–4)
- `train/dev_semeval_parids-labels.csv` — official split IDs with task-2 multi-hot category labels
- `task4_test.tsv` — unlabeled test set (no labels; used only for final submission)

We use the local `dont_patronize_me.py` module to load the TSV, then merge in the split IDs.

In [2]:
import sys
import pandas as pd
from ast import literal_eval

sys.path.insert(0, ".")  # so dont_patronize_me.py is found from the notebooks/ dir
from dont_patronize_me import DontPatronizeMe

TASK2_CATS = [
    "unbalanced_power", "shallow_solution", "presupposition",
    "authority_voice", "metaphor", "compassion", "poorer_merrier"
]

# Load full labeled corpus (binary label + orig_label 0-4)
dpm = DontPatronizeMe("../data", "../data/task4_test.tsv")
dpm.load_task1()
dpm.load_test()

corpus  = dpm.train_task1_df.copy()
test_df = dpm.test_set_df.copy()

# Load official train/dev splits (par_id + task-2 multi-hot labels)
def load_ids(path):
    df = pd.read_csv(path, dtype={"par_id": str})
    cats = pd.DataFrame(df["label"].apply(literal_eval).tolist(), columns=TASK2_CATS)
    return pd.concat([df[["par_id"]], cats], axis=1)

train_ids = load_ids("../data/train_semeval_parids-labels.csv")
dev_ids   = load_ids("../data/dev_semeval_parids-labels.csv")

# Merge to get text + binary label + task-2 categories in one DataFrame
corpus["par_id"] = corpus["par_id"].astype(str)
train_df = train_ids.merge(corpus, on="par_id")
dev_df   = dev_ids.merge(corpus, on="par_id")

print(f"Corpus : {len(corpus):,} rows")
print(f"Train  : {len(train_df):,} rows  |  PCL: {train_df.label.sum()} ({train_df.label.mean():.1%})")
print(f"Dev    : {len(dev_df):,}  rows  |  PCL: {dev_df.label.sum()} ({dev_df.label.mean():.1%})")
print(f"Test   : {len(test_df):,}  rows  (unlabeled)")

Corpus : 10,469 rows
Train  : 8,375 rows  |  PCL: 794 (9.5%)
Dev    : 2,094  rows  |  PCL: 199 (9.5%)
Test   : 3,832  rows  (unlabeled)
